# SLAYER-OCR: full-page pilot v1

Select a GPU runtime, then **Run all**. No uploads or HF token. Default: two historical development pages. Our mixed-v3 pipeline is compared with OvisOCR2; source PAGE regions with automatic line segmentation are a separate diagnostic.

Source transcriptions are **not verified gold**. No model promotion or SOTA claim is possible from this pilot. The 36-page historical test is excluded. Historical spelling is preserved. GPU execution of the new adapter is not yet verified.


## 1. Lightweight scoring dependencies


In [ ]:
%pip install jiwer==4.0.0 markdown-it-py==4.0.0 pillow==11.3.0


## 2. Frozen code and two-page scope
The runner is embedded with a SHA-256; existing OCR modules come from a pinned public Git commit.


In [ ]:
import hashlib
import importlib.util
import json
import os
from pathlib import Path
import subprocess
import sys
import venv

CONFIG = {'schema': 'slayer-full-page-pilot-config-v1', 'scope': 'historical Polish development diagnostics; not a gold benchmark or SOTA evaluation', 'target_review_pages': 100, 'available_non_test_pages': 80, 'default_inference_pages': 2, 'selection_salt': 'slayer-full-page-pilot-v1', 'dataset': {'repo': 'PiotrSty/impact-psnc-polish-ocr', 'revision': 'c7cb156fb95d2880699c33725bbaf1fbc1008fea', 'license': 'CC-BY-3.0', 'splits': {'validation': {'pages': 15, 'metadata_sha256': '536fc014963c2c55626e89f9bd07500826cb930f05ca3f8848352ca80c8856b1'}, 'train': {'pages': 65, 'metadata_sha256': '991ef94a2e2fb3ac519ec3582fa588b1a75a92a87b1b002468cf7c31d3c10427'}}, 'excluded_collections': ['NA2_FT', 'Nowiny_z_Rakuz_FT', 'Powodzenia_FT']}, 'models': {'mixed-v3': {'repo': 'PiotrSty/trocr-pl-mixed-v3', 'revision': '85d0c91c26f8e088849096dded7c9ba10b4cd9c9', 'weights_sha256': '19437f5769b8df8be5af1fb0c5bbcc3acccae9d889f337f0c4b9e94fa80d8bb7', 'packages': ['transformers==4.57.6', 'tokenizers==0.22.2', 'huggingface_hub==0.36.2', 'sentencepiece==0.2.1', 'pillow==11.3.0', 'opencv-python-headless==4.12.0.88', 'langdetect==1.0.9'], 'batch_size': 8, 'line_padding': 5}, 'ovis-ocr2': {'repo': 'ATH-MaaS/OvisOCR2', 'revision': '1fc9221b7823a371d6e97f92d527cc847e24e107', 'packages': ['transformers==5.18.0', 'pillow==11.3.0'], 'max_new_tokens': 8192, 'min_pixels': 200704, 'max_pixels': 8294400, 'adapter_status': 'Transformers SDPA portability pilot; GPU execution not yet verified', 'upstream_inference': 'https://huggingface.co/ATH-MaaS/OvisOCR2'}}, 'policy': {'reference_status': 'source-unreviewed', 'normalization': 'Unicode NFC + whitespace only', 'modernize_spelling': False, 'send_reference_text_to_model': False, 'final_test_used': False, 'automatic_model_promotion': False, 'automatic_publication': False, 'missing_and_error_predictions': 'score as empty text; retain error and raw output', 'oracle_geometry': 'diagnostic only; never mix oracle and automatic full-page results'}}
BASE_CODE_REVISION = 'ad8c3908a41949719a29ea259c5d286e078f788f'
RUNNER_SOURCE = '"""Stage and diagnose full-page OCR without promoting source labels to gold."""\nfrom __future__ import annotations\n\nimport argparse\nfrom collections import Counter, defaultdict\nimport hashlib\nfrom html.parser import HTMLParser\nimport importlib.metadata\nimport json\nfrom pathlib import Path, PurePosixPath\nimport sys\nimport time\nimport unicodedata\nfrom urllib.parse import quote\nfrom urllib.request import urlopen\nfrom xml.etree import ElementTree\n\n\ndef digest(path):\n    result = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            result.update(chunk)\n    return result.hexdigest()\n\n\ndef read_rows(path):\n    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines()\n            if line.strip()]\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\\n",\n                          encoding="utf-8")\n\n\ndef write_rows(path, rows):\n    Path(path).write_text("".join(json.dumps(row, ensure_ascii=False) + "\\n" for row in rows),\n                          encoding="utf-8", newline="\\n")\n\n\ndef safe_relative(value):\n    path = PurePosixPath(value)\n    if (not value or "\\\\" in value or ":" in value or path.is_absolute()\n            or ".." in path.parts):\n        raise ValueError(f"Unsafe relative path: {value}")\n    return path.as_posix()\n\n\ndef fetch(config, relative, target, expected, opener=urlopen):\n    target = Path(target)\n    if target.exists():\n        if digest(target) != expected:\n            raise ValueError(f"Cached checksum mismatch: {target}")\n        return target\n    source = config["dataset"]\n    url = (f\'https://huggingface.co/datasets/{source["repo"]}/resolve/\'\n           f\'{source["revision"]}/{quote(safe_relative(relative), safe="/")}\')\n    with opener(url, timeout=120) as response:\n        data = response.read()\n    if hashlib.sha256(data).hexdigest() != expected:\n        raise ValueError(f"Source checksum mismatch: {relative}")\n    target.parent.mkdir(parents=True, exist_ok=True)\n    target.write_bytes(data)\n    return target\n\n\ndef select_pages(rows, count, salt):\n    if not rows or len({row["id"] for row in rows}) != len(rows):\n        raise ValueError("Expected unique nonempty page IDs")\n    if not 1 <= count <= len(rows):\n        raise ValueError("Page count outside available non-test pool")\n    groups = defaultdict(list)\n    for row in rows:\n        groups[(row["split"], row["collection"])].append(row)\n    key = lambda value: hashlib.sha256((salt + "\\0" + value).encode()).hexdigest()\n    for group in groups.values():\n        group.sort(key=lambda row: key(row["id"]))\n    # Validation comes first for a bounded smoke run; never select by OCR score.\n    order = sorted(groups, key=lambda item: (item[0] != "validation", key(item[1])))\n    selected = []\n    while len(selected) < count:\n        for name in order:\n            if groups[name] and len(selected) < count:\n                selected.append(groups[name].pop(0))\n    return selected\n\n\ndef coordinates(element, width, height):\n    coords = element.find("./{*}Coords")\n    if coords is None:\n        return None\n    points = [(int(p.attrib["x"]), int(p.attrib["y"]))\n              for p in coords.findall("./{*}Point")]\n    if not points and coords.get("points"):\n        points = [tuple(map(int, value.split(","))) for value in coords.get("points").split()]\n    if not points:\n        return None\n    xs, ys = zip(*points)\n    if any(x < 0 or y < 0 or x > width or y > height for x, y in points):\n        return None\n    box = [min(xs), min(ys), min(width, max(xs) + 1), min(height, max(ys) + 1)]\n    if not (0 <= box[0] < box[2] <= width and 0 <= box[1] < box[3] <= height):\n        return None\n    return box\n\n\ndef inspect_reference(row, xml):\n    root = ElementTree.fromstring(xml)\n    page = root.find(".//{*}Page")\n    if page is None or (int(page.get("imageWidth")), int(page.get("imageHeight"))) != (\n            row["width"], row["height"]):\n        raise ValueError(f\'PAGE dimensions mismatch: {row["id"]}\')\n    order = {item.get("regionRef"): int(item.get("index"))\n             for item in root.findall(".//{*}RegionRefIndexed")}\n    regions = list(page.findall("./{*}TextRegion"))\n    regions.sort(key=lambda element: order.get(element.get("id"), 1_000_000))\n    lines, region_boxes, invalid_boxes, invalid_regions, content_regions = [], [], 0, 0, []\n    for region in regions:\n        direct = region.find("./{*}TextEquiv/{*}Unicode")\n        content = (direct.text or "") if direct is not None else "\\n".join(\n            node.text or "" for node in region.findall("./{*}TextLine/{*}TextEquiv/{*}Unicode"))\n        if not content.strip():\n            continue\n        content_regions.append(region)\n        region_box = coordinates(region, row["width"], row["height"])\n        if region_box is None:\n            invalid_regions += 1\n        else:\n            region_boxes.append({"id": region.get("id"), "bbox_xyxy": region_box,\n                                 "source_order_known": region.get("id") in order})\n        for line in region.findall("./{*}TextLine"):\n            box = coordinates(line, row["width"], row["height"])\n            if box is None:\n                invalid_boxes += 1\n            else:\n                lines.append({"id": line.get("id"), "bbox_xyxy": box,\n                              "region_id": region.get("id")})\n    text = row["text"]\n    flags = []\n    if "\\ufffd" in text:\n        flags.append("replacement-character")\n    if any(unicodedata.category(char) == "Co" for char in text):\n        flags.append("private-use-character")\n    if len("".join(text.split())) < 20:\n        flags.append("very-short-reference")\n    if not lines:\n        flags.append("no-source-line-geometry")\n    if not region_boxes:\n        flags.append("no-valid-region-geometry")\n    if invalid_boxes:\n        flags.append("invalid-line-geometry")\n    if invalid_regions:\n        flags.append("invalid-region-geometry")\n    if content_regions and any(region.get("id") not in order for region in content_regions):\n        flags.append("reading-order-incomplete")\n    return {"id": row["id"], "source_split": row["split"], "collection": row["collection"],\n            "reference_characters": len(text), "replacement_characters": text.count("\\ufffd"),\n            "private_use_characters": sum(unicodedata.category(c) == "Co" for c in text),\n            "text_regions": len(content_regions), "valid_line_boxes": len(lines),\n            "valid_region_boxes": len(region_boxes), "invalid_region_boxes": invalid_regions,\n            "empty_source_regions": len(regions) - len(content_regions),\n            "invalid_line_boxes": invalid_boxes, "flags": flags,\n            "completeness": "unverified", "source_regions": region_boxes}\n\n\ndef stage(config, output, count=None, opener=urlopen):\n    from PIL import Image\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    config_path = output / "config.json"\n    if config_path.exists() and json.loads(config_path.read_text(encoding="utf-8")) != config:\n        raise ValueError("Output belongs to a different configuration")\n    write_json(config_path, config)\n    rows = []\n    for split, spec in config["dataset"]["splits"].items():\n        if split not in ("train", "validation"):\n            raise ValueError("Final test must not be staged by this pilot")\n        path = fetch(config, f"pages/{split}/metadata.jsonl", output / "sources" / f"{split}.jsonl",\n                     spec["metadata_sha256"], opener)\n        current = read_rows(path)\n        if len(current) != spec["pages"] or any(row["split"] != split for row in current):\n            raise ValueError("Source metadata split/count mismatch")\n        rows.extend(current)\n    if len(rows) != config["available_non_test_pages"]:\n        raise ValueError("Non-test pool size changed")\n    for row in rows:\n        if (row["collection"] in config["dataset"]["excluded_collections"]\n                or row["license"] != config["dataset"]["license"]):\n            raise ValueError("Forbidden collection or license mismatch")\n    selected = select_pages(rows, len(rows) if count is None else count, config["selection_salt"])\n    selection_hash = hashlib.sha256(json.dumps([row["id"] for row in selected]).encode()).hexdigest()\n    if (output / "selection.json").exists():\n        old = json.loads((output / "selection.json").read_text(encoding="utf-8"))\n        if old["sha256"] != selection_hash:\n            raise ValueError("Use a new output for a different page selection")\n    write_json(output / "selection.json", {"ids": [row["id"] for row in selected], "sha256": selection_hash})\n    manifests, audits, inference = [], [], []\n    for index, row in enumerate(selected):\n        source = fetch(config, f\'pages/{row["split"]}/{safe_relative(row["file_name"])}\',\n                       output / "sources" / f"{index:04d}.jpg", row["image_sha256"], opener)\n        xml = fetch(config, row["pagexml"], output / "pagexml" / f"{index:04d}.xml",\n                    row["pagexml_sha256"], opener)\n        png = output / "images" / f"{index:04d}.png"\n        png.parent.mkdir(exist_ok=True)\n        with Image.open(source) as image:\n            image.seek(0)\n            if image.size != (row["width"], row["height"]):\n                raise ValueError(f\'Image dimensions mismatch: {row["id"]}\')\n            image.save(png, format="PNG")\n            with Image.open(png) as check:\n                if check.mode != image.mode or check.tobytes() != image.tobytes():\n                    raise ValueError("PNG pixel round-trip failed")\n        audit = inspect_reference(row, xml.read_bytes())\n        manifest = {**row, "image": png.relative_to(output).as_posix(), "sha256": digest(png),\n                    "source_sha256": row["image_sha256"], "reference_status": "source-unreviewed",\n                    "completeness_verified": False, "final_test": False,\n                    "pagexml_local": xml.relative_to(output).as_posix()}\n        manifests.append(manifest)\n        inference.append({key: manifest[key] for key in ("id", "image", "sha256", "width", "height")} |\n                         {"source_regions": audit.pop("source_regions")})\n        audits.append(audit)\n        print(f\'Staged {index + 1}/{len(selected)}: {row["id"]}\', flush=True)\n    write_rows(output / "manifest.jsonl", manifests)\n    write_rows(output / "inference-inputs.jsonl", inference)\n    counts = Counter(flag for audit in audits for flag in audit["flags"])\n    report = {"schema": "slayer-reference-audit-v1", "pages": len(selected),\n              "available_non_test_pages": len(rows), "target_review_pages": config["target_review_pages"],\n              "additional_pages_needed": max(0, config["target_review_pages"] - len(rows)),\n              "manifest_sha256": digest(output / "manifest.jsonl"),\n              "inference_inputs_sha256": digest(output / "inference-inputs.jsonl"),\n              "source_split_counts": dict(Counter(row["split"] for row in selected)),\n              "flag_counts": dict(counts), "results": audits, "gold_pages": 0,\n              "claim_boundary": "Flags are triage only; their absence does not establish completeness. "\n                                "Train pages may have been seen by the existing recognizer. No SOTA claim."}\n    write_json(output / "reference-audit.json", report)\n    return report\n\n\ndef validate_inputs(path):\n    path = Path(path)\n    rows = read_rows(path)\n    if not rows or len({row["id"] for row in rows}) != len(rows):\n        raise ValueError("Invalid inference page IDs")\n    for row in rows:\n        if (set(row) != {"id", "image", "sha256", "width", "height", "source_regions"}\n                or any(set(region) != {"id", "bbox_xyxy", "source_order_known"}\n                       for region in row["source_regions"])):\n            raise ValueError("Reference text is forbidden in model inputs")\n        image = path.parent / safe_relative(row["image"])\n        if digest(image) != row["sha256"]:\n            raise ValueError(f\'Input checksum mismatch: {row["id"]}\')\n    return rows\n\n\ndef load_mixed(spec):\n    from huggingface_hub import snapshot_download\n    from ocr.config import OcrConfig\n    from ocr.detector import sort_reading_order, sort_reading_order_columns\n    from ocr.opencv_detector import OpenCVDetector\n    from ocr.preprocess import load_image\n    from ocr.recognizer import Recognizer\n    model_dir = Path(snapshot_download(spec["repo"], revision=spec["revision"]))\n    if digest(model_dir / "model.safetensors") != spec["weights_sha256"]:\n        raise ValueError("mixed-v3 weights checksum mismatch")\n    config = OcrConfig(recognizer_pl=str(model_dir), recognizer_en=str(model_dir),\n                       recognizer_fallback=str(model_dir), recognizer_backend="trocr",\n                       use_fallback_if_pl_missing=False, device="cuda", force_language="pl",\n                       deskew=False, correct_text=False, batch_size=spec["batch_size"],\n                       line_padding=spec["line_padding"])\n    detector, recognizer = OpenCVDetector(config), Recognizer(config)\n\n    def infer(path, page):\n        image = load_image(path)\n        boxes = detector.detect(image)\n        decoded = recognizer.recognize_lines(image, boxes, ["pl"] * len(boxes))\n        if len(decoded) != len(boxes):\n            raise ValueError("Recognizer output count mismatch")\n        texts = {box.to_tuple(): text for box, (text, _) in zip(boxes, decoded)}\n        row_major = sort_reading_order(boxes, image.shape[0])\n        columns = sort_reading_order_columns(boxes, image.shape[1], image.shape[0])\n        results = {"mixed-v3-row-major": {"text": "\\n".join(texts[box.to_tuple()] for box in row_major)},\n                   "mixed-v3-column-order": {"text": "\\n".join(texts[box.to_tuple()] for box in columns)}}\n        if page["source_regions"]:\n            region_texts, detected_region_lines = [], 0\n            for region in page["source_regions"]:\n                x1, y1, x2, y2 = region["bbox_xyxy"]\n                crop = image[y1:y2, x1:x2]\n                local_boxes = sort_reading_order(detector.detect(crop), crop.shape[0])\n                recognized = recognizer.recognize_lines(crop, local_boxes, ["pl"] * len(local_boxes))\n                if len(recognized) != len(local_boxes):\n                    raise ValueError("Region recognizer output count mismatch")\n                detected_region_lines += len(local_boxes)\n                region_texts.append("\\n".join(text for text, _ in recognized))\n            results["mixed-v3-source-regions"] = {"text": "\\n".join(region_texts),\n                                                   "region_detected_lines": detected_region_lines}\n        else:\n            results["mixed-v3-source-regions"] = {"status": "error", "text": "",\n                                                   "error": "No valid source region geometry"}\n        for result in results.values():\n            result["detected_lines"] = len(boxes)\n        return results\n\n    return infer\n\n\nOVIS_PROMPT = ("Extract all readable content from the image in natural human reading order "\n               "and output a single Markdown document. Represent tables as HTML and formulas "\n               "as LaTeX. Represent visual regions with an HTML image tag. Preserve original "\n               "text without translation, paraphrasing, spelling correction or modernization. "\n               "Preserve historical characters including \\u00e1 and \\u017f.")\n\n\ndef load_ovis(spec):\n    import torch\n    from PIL import Image\n    from transformers import AutoModelForImageTextToText, AutoProcessor\n    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n    model = AutoModelForImageTextToText.from_pretrained(\n        spec["repo"], revision=spec["revision"], torch_dtype=dtype,\n        attn_implementation="sdpa").to("cuda").eval()\n    processor = AutoProcessor.from_pretrained(spec["repo"], revision=spec["revision"])\n\n    def infer(path, page):\n        with Image.open(path) as source:\n            image = source.convert("RGB")\n        messages = [{"role": "user", "content": [\n            {"type": "image", "image": image}, {"type": "text", "text": OVIS_PROMPT}]}]\n        inputs = processor.apply_chat_template(\n            messages, tokenize=True, add_generation_prompt=True, return_dict=True,\n            return_tensors="pt", enable_thinking=False,\n            images_kwargs={"min_pixels": spec["min_pixels"], "max_pixels": spec["max_pixels"]})\n        inputs = inputs.to(model.device)\n        with torch.inference_mode():\n            outputs = model.generate(**inputs, max_new_tokens=spec["max_new_tokens"], do_sample=False)\n        generated = outputs[0][inputs["input_ids"].shape[1]:]\n        raw = processor.decode(generated, skip_special_tokens=True,\n                               clean_up_tokenization_spaces=False)\n        return {"ovis-ocr2-full-page": {"text": raw, "format": "markdown",\n                "generated_tokens": len(generated),\n                "token_limit_reached": len(generated) >= spec["max_new_tokens"]}}\n\n    return infer\n\n\ndef run_worker(engine, config, inputs, output):\n    import torch\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    rows = validate_inputs(inputs)\n    spec = config["models"][engine]\n    identity = {"engine": engine, "spec": spec, "input_sha256": digest(inputs),\n                "runner_sha256": digest(__file__)}\n    identity_file = output / "identity.json"\n    if identity_file.exists() and json.loads(identity_file.read_text(encoding="utf-8")) != identity:\n        raise ValueError("Existing predictions belong to different inputs/code/model")\n    write_json(identity_file, identity)\n    if not torch.cuda.is_available():\n        raise RuntimeError("Select a GPU runtime in Colab")\n    torch.manual_seed(42)\n    variants = (["mixed-v3-row-major", "mixed-v3-column-order", "mixed-v3-source-regions"]\n                if engine == "mixed-v3" else ["ovis-ocr2-full-page"])\n    existing = {}\n    for variant in variants:\n        path = output / f"{variant}.jsonl"\n        records = read_rows(path) if path.exists() else []\n        if len({row["id"] for row in records}) != len(records):\n            raise ValueError("Duplicate checkpoint prediction")\n        if set(row["id"] for row in records) - set(row["id"] for row in rows):\n            raise ValueError("Unknown checkpoint page")\n        existing[variant] = {row["id"] for row in records}\n    pending = [row for row in rows if any(row["id"] not in existing[v] for v in variants)]\n    load_error = None\n    started = time.perf_counter()\n    try:\n        infer = (load_mixed if engine == "mixed-v3" else load_ovis)(spec) if pending else None\n    except Exception as exc:\n        infer, load_error = None, f"{type(exc).__name__}: {exc}"\n    load_seconds = time.perf_counter() - started\n    for index, page in enumerate(pending, 1):\n        started = time.perf_counter()\n        torch.cuda.reset_peak_memory_stats()\n        try:\n            if load_error:\n                raise RuntimeError(load_error)\n            torch.cuda.synchronize()\n            predictions = infer(Path(inputs).parent / page["image"], page)\n            torch.cuda.synchronize()\n        except Exception as exc:\n            predictions = {variant: {"status": "error", "text": "",\n                                     "error": f"{type(exc).__name__}: {exc}"} for variant in variants}\n        elapsed = time.perf_counter() - started\n        for variant in variants:\n            if page["id"] in existing[variant]:\n                continue\n            record = {"id": page["id"], "status": "ok", "elapsed_seconds": elapsed,\n                      "elapsed_scope": "all variants for this engine/page; not per-variant latency",\n                      "peak_allocated_bytes": torch.cuda.max_memory_allocated(), **predictions[variant]}\n            with (output / f"{variant}.jsonl").open("a", encoding="utf-8", newline="\\n") as stream:\n                stream.write(json.dumps(record, ensure_ascii=False) + "\\n")\n                stream.flush()\n        print(f\'{engine} {index}/{len(pending)}: {page["id"]}\', flush=True)\n    environment = {"gpu": torch.cuda.get_device_name(0), "python": sys.version,\n                   "model_load_seconds": load_seconds, "load_error": load_error,\n                   "reference_text_sent_to_model": False, "packages": {}}\n    for package in ("torch", "transformers", "huggingface_hub", "tokenizers", "Pillow"):\n        environment["packages"][package] = importlib.metadata.version(package)\n    write_json(output / "environment.json", environment)\n\n\nclass TextHTMLParser(HTMLParser):\n    def __init__(self):\n        super().__init__(convert_charrefs=True)\n        self.parts = []\n        self.hidden = 0\n\n    def handle_starttag(self, tag, attrs):\n        if tag in ("script", "style"):\n            self.hidden += 1\n        if tag in ("p", "br", "td", "th", "tr", "div"):\n            self.parts.append(" ")\n\n    def handle_endtag(self, tag):\n        if tag in ("script", "style"):\n            self.hidden = max(0, self.hidden - 1)\n        if tag in ("p", "td", "th", "tr", "div"):\n            self.parts.append(" ")\n\n    def handle_data(self, value):\n        if not self.hidden:\n            self.parts.append(value)\n\n\ndef markdown_text(value):\n    from markdown_it import MarkdownIt\n    # Parse presentation syntax only. Do not repair letters, words or OCR repeats.\n    html = MarkdownIt("commonmark", {"html": True}).render(value)\n    parser = TextHTMLParser()\n    parser.feed(html)\n    return "".join(parser.parts)\n\n\ndef score(manifest, prediction_root, output):\n    from training.benchmark_pages import evaluate\n    rows = read_rows(manifest)\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    reports = {}\n    for source in sorted(Path(prediction_root).glob("*/*.jsonl")):\n        variant = source.stem\n        projected = []\n        for row in read_rows(source):\n            projected.append({**row, "raw_text": row["text"],\n                              "text": markdown_text(row["text"]) if row.get("format") == "markdown"\n                              else row["text"]})\n        path = output / f"{variant}.jsonl"\n        write_rows(path, projected)\n        report = evaluate(manifest, path)\n        report["cer_macro"] = sum(item["cer"] for item in report["results"]) / report["pages"]\n        report["token_limit_pages"] = sum(item.get("token_limit_reached", False) for item in projected)\n        report["oracle_geometry"] = "source-regions" in variant\n        # This stage has no human completeness attestations; never silently promote it.\n        report["reference_status"] = "source-unreviewed"\n        report["eligible_for_model_promotion"] = False\n        report["groups"] = {}\n        for key in ("split", "collection"):\n            for group in sorted({row[key] for row in rows}):\n                group_hash = hashlib.sha256(f"{variant}:{key}:{group}".encode()).hexdigest()[:16]\n                subset = Path(manifest).parent / f".pilot-score-{group_hash}.jsonl"\n                supplied = {item["id"]: item for item in projected}\n                selected = [row for row in rows if row[key] == group]\n                predictions = output / f".subset-{variant}.jsonl"\n                try:\n                    write_rows(subset, selected)\n                    write_rows(predictions, [supplied[row["id"]] for row in selected if row["id"] in supplied])\n                    group_report = evaluate(subset, predictions)\n                    report["groups"][f"{key}:{group}"] = {field: group_report[field]\n                                                         for field in ("pages", "cer_micro", "wer_micro", "errors_or_missing")}\n                finally:\n                    subset.unlink(missing_ok=True)\n                    predictions.unlink(missing_ok=True)\n        reports[variant] = report\n    if not reports:\n        raise ValueError("No prediction artifacts found")\n    summary = {"schema": "slayer-full-page-pilot-result-v1", "pages": len(rows),\n               "reports": reports, "model_promotion": False, "sota_claim": False,\n               "normalization": "Unicode NFC + whitespace only",\n               "projection": "CommonMark to HTML to visible text; raw outputs retained",\n               "claim_boundary": "Unreviewed source references, possible training exposure, "\n                                 "small development sample. Oracle and automatic results are separate."}\n    write_json(output / "metrics.json", summary)\n    return summary\n\n\ndef package(work):\n    work = Path(work)\n    files = [path for path in work.rglob("*") if path.is_file()\n             and not any(part.startswith(".") for part in path.relative_to(work).parts)\n             and path.suffix in (".json", ".jsonl", ".log")\n             and "model" not in path.relative_to(work).parts]\n    import zipfile\n    archive = work / "full-page-pilot-v1-evidence.zip"\n    checksums = {path.relative_to(work).as_posix(): digest(path) for path in files}\n    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as stream:\n        for path in files:\n            stream.write(path, path.relative_to(work).as_posix())\n        stream.writestr("checksums.json", json.dumps(checksums, indent=2))\n    return archive\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("command", choices=("stage", "worker"))\n    parser.add_argument("--config", required=True)\n    parser.add_argument("--output", required=True)\n    parser.add_argument("--pages", type=int)\n    parser.add_argument("--engine", choices=("mixed-v3", "ovis-ocr2"))\n    parser.add_argument("--inputs")\n    args = parser.parse_args()\n    config = json.loads(Path(args.config).read_text(encoding="utf-8"))\n    if args.command == "stage":\n        report = stage(config, args.output, args.pages)\n        print(json.dumps({key: value for key, value in report.items() if key != "results"}, indent=2))\n    else:\n        if not args.engine or not args.inputs:\n            parser.error("worker requires --engine and --inputs")\n        run_worker(args.engine, config, args.inputs, args.output)\n\n\nif __name__ == "__main__":\n    main()\n'
RUNNER_SHA256 = '13c60d5ca4eabe1a805c0943dfb7a50e5f0b1b18069c53fd6a264a6fa6eed254'
PAGES = CONFIG['default_inference_pages']  # 2-page smoke; increase only after reviewing its evidence.
WORK = Path('/content/slayer-full-page-pilot-v1')
WORK.mkdir(parents=True, exist_ok=True)
config_path = WORK / 'config.json'
if config_path.exists():
    assert json.loads(config_path.read_text()) == CONFIG, 'Use a new WORK for changed configuration.'
config_path.write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')
runner_path = WORK / 'full_page_pilot.py'
assert hashlib.sha256(RUNNER_SOURCE.encode()).hexdigest() == RUNNER_SHA256
if runner_path.exists():
    assert hashlib.sha256(runner_path.read_bytes()).hexdigest() == RUNNER_SHA256, 'Use a new WORK for changed runner.'
runner_path.write_text(RUNNER_SOURCE, encoding='utf-8', newline='\n')

repo = Path('/content/OCR_engine-full-page-pilot')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/PiotrStyla/OCR_engine.git', str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', BASE_CODE_REVISION], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', BASE_CODE_REVISION], check=True)
assert subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip() == BASE_CODE_REVISION, 'Dedicated checkout has changed; use a fresh session.'
sys.path.insert(0, str(repo))
spec = importlib.util.spec_from_file_location('full_page_pilot', runner_path)
pilot = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pilot)
pilot.write_json(WORK / 'code-provenance.json', {
    'base_code_revision': BASE_CODE_REVISION, 'embedded_runner_sha256': RUNNER_SHA256,
    'gpu_inference_previously_verified': False,
    'environment_bootstrap': 'venv-without-pip-host-pip-python-v2',
})
print('WORK:', WORK)
print('PAGES:', PAGES)


## 3. Automatic HF download and reference audit


In [ ]:
dataset = WORK / 'dataset'
audit = pilot.stage(CONFIG, dataset, PAGES)
print(json.dumps({key: value for key, value in audit.items() if key != 'results'}, indent=2))
assert audit['gold_pages'] == 0
print('SOURCE REFERENCES ARE UNREVIEWED. These scores cannot select a production model.')


## 4. Separate model environments
Each engine has its own Transformers installation and subprocess; the notebook kernel is not used for model loading. Environment creation does not require ensurepip. Rerunning this cell repairs partial environment creation without deleting data.


In [ ]:
import os
from pathlib import Path
import subprocess
import sys
import venv


def prepare_pilot_environment(environment):
    environment = Path(environment)
    # Colab can lack ensurepip; repair a partial venv without deleting its files.
    venv.EnvBuilder(with_pip=False, system_site_packages=True).create(environment)
    python = environment / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
    prefix = subprocess.check_output(
        [str(python), '-c', 'import sys; print(sys.prefix)'], text=True).strip()
    if Path(prefix).resolve() != environment.resolve():
        raise RuntimeError('Refusing installation outside the requested model environment.')
    return python


def pilot_pip_command(python, *arguments):
    # Host pip manages the target interpreter even when its venv has no local pip.
    return [sys.executable, '-m', 'pip', '--python', str(python), *arguments]

import importlib.metadata
import torch
assert torch.cuda.is_available(), 'Colab: Runtime > Change runtime type > GPU.'
assert tuple(int(part) for part in importlib.metadata.version('pip').split('.')[:2]) >= (22, 3), 'Host pip >= 22.3 is required for --python.'
print('GPU:', torch.cuda.get_device_name(0), 'Torch:', torch.__version__)
python_by_engine = {}
for engine, model_spec in CONFIG['models'].items():
    environment = Path('/content/slayer-full-page-pilot-envs') / engine
    python = prepare_pilot_environment(environment)
    # Install into isolated venvs, never replace Transformers in the notebook kernel.
    subprocess.run(pilot_pip_command(python, 'install', '--no-cache-dir',
                                    *model_spec['packages']), check=True)
    required = {item.split('==')[0]: item.split('==')[1] for item in model_spec['packages']}
    check = ('import importlib.metadata,json; required=' + repr(required) +
             '; actual={k:importlib.metadata.version(k) for k in required}; '
             'assert actual==required,(actual,required); '
             'from transformers.generation import GenerationMixin; '
             'print(json.dumps(actual))')
    if engine == 'ovis-ocr2':
        check += '; from transformers import AutoModelForImageTextToText, AutoProcessor'
    subprocess.run([str(python), '-c', check], check=True)
    python_by_engine[engine] = python


## 5. Inference with per-page checkpoints
Rerunning preserves recorded pages, including errors. To retry failed pages, use a new WORK directory. If a later cell fails, the download cell still packages available diagnostics.


In [ ]:
process_env = {**os.environ, 'PYTHONPATH': str(repo), 'TOKENIZERS_PARALLELISM': 'false'}
process_results = {}
try:
    for engine, python in python_by_engine.items():
        destination = WORK / 'predictions' / engine
        destination.mkdir(parents=True, exist_ok=True)
        with (WORK / (engine + '.log')).open('w', encoding='utf-8') as log:
            process = subprocess.Popen([
                str(python), '-u', str(runner_path), 'worker', '--config', str(config_path),
                '--engine', engine, '--inputs', str(dataset / 'inference-inputs.jsonl'),
                '--output', str(destination),
            ], env=process_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            try:
                for line in process.stdout:
                    print(line, end='')
                    log.write(line)
                    log.flush()
                process_results[engine] = process.wait()
            finally:
                if process.poll() is None:
                    process.terminate()
                    process.wait()
        print(engine, 'exit status:', process_results[engine])
finally:
    pilot.write_json(WORK / 'process-results.json', process_results)
    evidence_zip = pilot.package(WORK)
    print('PARTIAL_EVIDENCE_READY:', evidence_zip)


## 6. Provisional CER/WER
Missing or failed predictions are scored as empty. Raw Markdown is retained; only presentation markup is projected to text. Source-region crops are diagnostic annotation-assisted inputs, not automatic layout predictions. Tables, reading-order accuracy and hallucination rates require a later reviewed benchmark and are not claimed here.


In [ ]:
summary = pilot.score(dataset / 'manifest.jsonl', WORK / 'predictions', WORK / 'scores')
for label, report in summary['reports'].items():
    print(f"{label}: CER={report['cer_micro']:.2%}, WER={report['wer_micro']:.2%}, "
          f"errors={report['errors_or_missing']}, token_limit_pages={report['token_limit_pages']}, "
          f"oracle={report['oracle_geometry']}")
print('DEVELOPMENT DIAGNOSTICS ONLY. SOTA claim:', summary['sota_claim'])

import matplotlib.pyplot as plt
labels = list(summary['reports'])
values = [summary['reports'][label]['cer_micro'] * 100 for label in labels]
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(labels, values)
ax.set_xlim(left=0)
ax.set_xlabel('CER (%) - unreviewed source references')
ax.set_title(f'Full-page pilot: {summary["pages"]} development pages; oracle is diagnostic')
for index, value in enumerate(values):
    ax.text(value, index, f' {value:.1f}%', va='center')
fig.tight_layout()
fig.savefig(WORK / 'cer-diagnostic.png', dpi=140)
plt.show()


## 7. Download evidence
Run this cell even after an inference or scoring error. Send back the resulting ZIP.


In [ ]:
from google.colab import files
evidence_zip = pilot.package(WORK)
print('Evidence:', evidence_zip)
print('Includes raw predictions, metrics, input provenance, errors and package versions; no scans or weights.')
files.download(str(evidence_zip))
